# Support search: sparse versus dense, recall and index budgets

A support team wants search over its help notes. The exercises in this module built the pieces
(keyword vectors, cosine, exact filtered top-k, index recall). This notebook extends them into the
measurements you would make before choosing a search design:

1. Implement BM25 and compare it with dense retrieval on a small labelled query set.
2. Average index recall over many queries, skipping and reporting queries with nothing to find.
3. Estimate index memory for flat float32 storage and for product quantisation.
4. Simulate an IVF index and sweep `nprobe` to trace recall against vectors scanned.
5. Interpret the results.

Everything runs offline with NumPy and the standard library. The "dense" vectors are small
hand-made stand-ins for encoder output (4 topic axes), so results are reproducible without a model.
Fill in each TODO cell, then run the check cell after it. Restart the kernel and run all cells before
submitting.

In [ ]:
import math
from collections import Counter

import numpy as np

## 1. BM25 versus dense retrieval

The corpus has nine notes. Each note also has a dense vector on four axes
(billing, account access, shipping, errors) standing in for a text-embedding model.

Write `bm25_scores(query, docs, k1=1.2, b=0.75)` returning one float per document, in corpus order:

- tokenise query and documents with `text.casefold().split()`;
- `idf = ln(1 + (N - df + 0.5) / (df + 0.5))`, where `df` counts documents containing the term;
- each distinct query term present in a document adds `idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * len(doc) / avgdl))`;
- a document that shares no query term scores 0.0.

The helpers `rank_ids` (sort by (−score, id)) and `relevance_recall` are provided.

In [ ]:
NOTES = {
    "n1": ("refund to card for a double charge", [0.9, 0.0, 0.1, 0.1]),
    "n2": ("card declined error e1234 at checkout", [0.5, 0.0, 0.0, 0.8]),
    "n3": ("reset your password from the login page", [0.0, 0.95, 0.0, 0.1]),
    "n4": ("two factor codes not arriving by sms", [0.0, 0.9, 0.0, 0.3]),
    "n5": ("track a parcel that has not shipped", [0.0, 0.0, 0.95, 0.1]),
    "n6": ("change the delivery address before dispatch", [0.1, 0.0, 0.9, 0.0]),
    "n7": ("app crashes with error e2001 on startup", [0.0, 0.1, 0.0, 0.95]),
    "n8": ("cancel your subscription and stop billing", [0.9, 0.2, 0.0, 0.0]),
    "n9": ("what an error message means and who to contact", [0.0, 0.1, 0.1, 0.95]),
}
# query text, query's stand-in dense vector, set of relevant note IDs (human labels)
QUERIES = {
    "code":     ("error e1234", [0.1, 0.0, 0.0, 1.0], {"n2"}),
    "login":    ("i cannot log in", [0.0, 1.0, 0.0, 0.1], {"n3", "n4"}),
    "plan":     ("end my plan", [0.9, 0.1, 0.0, 0.0], {"n8"}),
    "refund":   ("refund card", [1.0, 0.0, 0.0, 0.0], {"n1"}),
    "parcel":   ("parcel not shipped", [0.0, 0.0, 1.0, 0.0], {"n5"}),
    "delivery": ("where is my delivery", [0.0, 0.0, 1.0, 0.0], {"n5", "n6"}),
}
IDS = list(NOTES)

def rank_ids(scores, ids):
    """IDs sorted by descending score, ties broken by ascending ID."""
    return [i for s, i in sorted(zip(scores, ids), key=lambda p: (-p[0], p[1]))]

def relevance_recall(ranked, relevant, k):
    """Share of the human-labelled relevant IDs that appear in the top k."""
    return len(set(ranked[:k]) & set(relevant)) / len(relevant)

def dense_scores(qvec, vectors):
    q = np.asarray(qvec, float)
    V = np.asarray(vectors, float)
    return list((V @ q) / (np.linalg.norm(V, axis=1) * np.linalg.norm(q)))

In [ ]:
def bm25_scores(query, docs, k1=1.2, b=0.75):
    # TODO: return one BM25 score per document, in corpus order.
    raise NotImplementedError

In [ ]:
lesson_docs = ["refund to card", "card declined error e1234 at checkout",
               "refund refund refund policy for orders shipped late", "reset your password"]
got = bm25_scores("card refund", lesson_docs)
assert len(got) == 4, "Return one score per document"
np.testing.assert_allclose(got, [1.658, 0.641, 0.965, 0.0], atol=1e-3,
                           err_msg="Scores should match the lesson's worked example")
np.testing.assert_allclose(bm25_scores("refund card", lesson_docs), got, err_msg="Query word order must not matter")
np.testing.assert_allclose(bm25_scores("card card", lesson_docs), bm25_scores("card", lesson_docs),
                           err_msg="Count each distinct query term once")
assert bm25_scores("Refund", ["refund now", "other text"])[0] > 0, "Case-fold before matching"
flat = bm25_scores("refund", lesson_docs, b=0.0)
assert flat[0] < flat[2], "With b = 0 there is no length penalty, so three repeats should beat one"
print("OK")

In [ ]:
docs = [NOTES[i][0] for i in IDS]
vecs = [NOTES[i][1] for i in IDS]
K = 2
results = {}
print(f"{'query':<10}{'BM25 top-2':<14}{'dense top-2':<14}{'BM25 R@2':>9}{'dense R@2':>10}")
for name, (text, qvec, relevant) in QUERIES.items():
    sparse = rank_ids(bm25_scores(text, docs), IDS)
    dense = rank_ids(dense_scores(qvec, vecs), IDS)
    results[name] = (relevance_recall(sparse, relevant, K), relevance_recall(dense, relevant, K))
    print(f"{name:<10}{','.join(sparse[:K]):<14}{','.join(dense[:K]):<14}"
          f"{results[name][0]:>9.2f}{results[name][1]:>10.2f}")
print("mean      ", " " * 26, f"{np.mean([r[0] for r in results.values()]):>9.2f}",
      f"{np.mean([r[1] for r in results.values()]):>9.2f}")

In [ ]:
code_sparse = rank_ids(bm25_scores("error e1234", docs), IDS)
assert code_sparse[0] == "n2", "BM25 should put the note containing the rare code e1234 first"
assert results["code"] == (1.0, 0.0), "On the code query BM25 finds n2 and the dense stand-in does not"
assert results["login"][0] == 0.0, "BM25 shares no word with 'i cannot log in' ('login' is a different token)"
assert results["login"][1] == 1.0, "The dense stand-in should find both access notes"
print("OK")

## 2. Index recall over many queries

Write `mean_index_recall(exact_runs, approx_runs)`. Both arguments are lists with one ID list per query,
in the same query order. For each query, recall is the number of **distinct** exact IDs found in the
approximate list divided by the number of distinct exact IDs. Skip queries whose exact list is empty.

Return a tuple `(mean_recall, n_used, n_skipped)`, with `mean_recall = None` when every query was
skipped. Raise `ValueError` if the two lists have different lengths.

In [ ]:
def mean_index_recall(exact_runs, approx_runs):
    # TODO
    raise NotImplementedError

In [ ]:
m, used, skipped = mean_index_recall([["a", "b"], ["c"], []], [["a", "x"], ["c"], ["z"]])
assert (used, skipped) == (2, 1), "Two queries count; the empty exact list is skipped"
assert math.isclose(m, 0.75), "Mean of 0.5 and 1.0 is 0.75; the skipped query must not count as 0"
m, used, skipped = mean_index_recall([["a", "b"]], [["a", "a", "a"]])
assert math.isclose(m, 0.5), "Duplicate approximate IDs count once"
assert mean_index_recall([[], []], [["a"], []]) == (None, 0, 2), "All skipped gives (None, 0, 2)"
assert mean_index_recall([["a"], ["b", "c", "d", "e"]], [[], ["e", "d"]])[0] == 0.25, "Average per-query recall (0 and 0.5)"
try:
    mean_index_recall([["a"]], [])
except ValueError:
    pass
else:
    raise AssertionError("Reject runs of different lengths")
print("OK")

## 3. Index memory estimate

Write two functions returning a whole number of bytes:

- `flat_index_bytes(n, d, bytes_per_value=4)`: n vectors of d values (4 for float32, 2 for float16, 1 for int8).
- `pq_index_bytes(n, d, m)`: product quantisation with m sub-vectors and 8-bit codes. Each vector stores m
  one-byte codes, and the index also stores m codebooks of 256 float32 centroids of length d // m.
  Raise `ValueError` if d is not divisible by m.

Then print the size in decimal GB of 1,000,000 vectors at d = 768 for float32, int8 and PQ with m = 96.

In [ ]:
def flat_index_bytes(n, d, bytes_per_value=4):
    # TODO
    raise NotImplementedError

def pq_index_bytes(n, d, m):
    # TODO
    raise NotImplementedError

In [ ]:
for label, size in [("float32", flat_index_bytes(1_000_000, 768)),
                    ("int8", flat_index_bytes(1_000_000, 768, 1)),
                    ("PQ m=96", pq_index_bytes(1_000_000, 768, 96))]:
    print(f"{label:<8} {size / 1e9:.3f} GB")

In [ ]:
assert flat_index_bytes(1_000_000, 768) == 3_072_000_000, "1M x 768 float32 is 3,072,000,000 bytes"
assert flat_index_bytes(1_000_000, 384, 2) == 768_000_000, "float16 uses 2 bytes per value"
assert flat_index_bytes(10, 3, 1) == 30, "int8 uses 1 byte per value"
assert pq_index_bytes(1_000_000, 768, 96) == 96_000_000 + 786_432, "Codes are n*m bytes; codebooks are m*256*(d/m)*4 bytes"
assert pq_index_bytes(1_000, 64, 8) == 8_000 + 65_536, "Check the PQ formula on a second size"
try:
    pq_index_bytes(10, 768, 100)
except ValueError:
    pass
else:
    raise AssertionError("Reject d not divisible by m")
print("OK")

## 4. Simulated IVF: recall versus vectors scanned

The next cell builds 5,000 unit vectors and "trains" an IVF index with 50 centroids by k-means
(as in the lesson). `assign[i]` is the cluster of vector i.

Write `ivf_search(q, X, C, assign, nprobe, k)`:

- pick the `nprobe` clusters whose centroids have the highest dot product with `q` (nearest);
- score every vector in those clusters by dot product with `q`;
- return `(ids, n_scanned)`: up to k vector indices sorted by (−score, index), and how many vectors were scored.

Then write `nprobe_sweep(Q, X, C, assign, nprobes, k)`. Compute each query's exact top-k by brute force with
the same (−score, index) order, run `ivf_search` for every value in `nprobes`, and return a list of
`(nprobe, mean_recall, mean_fraction_scanned)` tuples, using your `mean_index_recall`.
Finally, `choose_nprobe(sweep, target)` returns the smallest nprobe whose mean recall reaches `target`,
or None.

In [ ]:
rng = np.random.default_rng(7)
n, d, nlist, k = 5_000, 32, 50, 10
centers = rng.normal(size=(10, d))
X = centers[rng.integers(10, size=n)] + 1.0 * rng.normal(size=(n, d))
X /= np.linalg.norm(X, axis=1, keepdims=True)
Q = X[rng.choice(n, 100, replace=False)] + 0.1 * rng.normal(size=(100, d))
Q /= np.linalg.norm(Q, axis=1, keepdims=True)

C = X[rng.choice(n, nlist, replace=False)]
for _ in range(10):
    assign = np.argmax(X @ C.T, axis=1)
    C = np.stack([X[assign == j].mean(axis=0) if np.any(assign == j) else C[j] for j in range(nlist)])
    C /= np.linalg.norm(C, axis=1, keepdims=True)
assign = np.argmax(X @ C.T, axis=1)
print("cluster sizes: min", np.bincount(assign, minlength=nlist).min(),
      "max", np.bincount(assign, minlength=nlist).max())

In [ ]:
def ivf_search(q, X, C, assign, nprobe, k):
    # TODO
    raise NotImplementedError

def nprobe_sweep(Q, X, C, assign, nprobes, k):
    # TODO
    raise NotImplementedError

def choose_nprobe(sweep, target):
    # TODO
    raise NotImplementedError

In [ ]:
Xs = np.array([[1.0, 0.0], [0.8, 0.6], [0.0, 1.0], [0.6, 0.8]])
Cs = np.array([[1.0, 0.0], [0.0, 1.0]])
As = np.array([0, 0, 1, 1])
ids, scanned = ivf_search(np.array([0.6, 0.8]), Xs, Cs, As, 1, 2)
assert (ids, scanned) == ([3, 2], 2), "nprobe=1 should scan only cluster 1 (nearest centroid) and rank 3 above 2"
ids, scanned = ivf_search(np.array([0.6, 0.8]), Xs, Cs, As, 2, 2)
assert (ids, scanned) == ([3, 1], 4), "nprobe=2 scans all four vectors and finds the true top 2"
ids, _ = ivf_search(np.array([1.0, 0.0]), np.array([[1.0, 0.0], [1.0, 0.0]]), Cs, np.array([0, 0]), 1, 2)
assert ids == [0, 1], "Break score ties by ascending index"

sweep = nprobe_sweep(Q, X, C, assign, [1, 2, 4, 8, 16, nlist], k)
for p, r, f in sweep:
    print(f"nprobe={p:>2}  recall@{k}={r:.3f}  scanned={f:.1%}")
recalls = [r for _, r, _ in sweep]
fracs = [f for _, _, f in sweep]
assert all(a <= b + 1e-12 for a, b in zip(recalls, recalls[1:])), "Probing more clusters can never lower recall"
assert all(a <= b for a, b in zip(fracs, fracs[1:])), "Probing more clusters scans more vectors"
assert math.isclose(recalls[-1], 1.0) and math.isclose(fracs[-1], 1.0), "Probing every cluster is exact search"
assert recalls[0] < 0.95, "One cluster should miss some neighbours on this data"
best = choose_nprobe(sweep, 0.95)
assert best == min(p for p, r, _ in sweep if r >= 0.95), "Pick the smallest nprobe that reaches the target"
assert choose_nprobe([(1, 0.5, 0.1), (4, 0.9, 0.3)], 0.95) is None, "Return None when no setting reaches the target"
assert choose_nprobe([(2, 0.97, 0.1), (1, 0.96, 0.05)], 0.95) == 1, "Do not assume the sweep is sorted"
print("chosen nprobe for recall >= 0.95:", best)
print("OK")

## 5. Interpretation

Answer in 150 to 300 words:

- From task 1, name one query where BM25 beat the dense stand-in and one where it lost, and explain each
  in terms of what sparse and dense vectors can and cannot represent. What would hybrid search do here?
- From task 4, which `nprobe` would you ship for a 0.95 recall target, and what fraction of the corpus
  does it scan? What would you measure next before trusting it on real embeddings?
- From task 3, would 1,000,000 float32 vectors at d = 768 fit comfortably on an 8 GB laptop? Which option
  would you choose and what does it cost?
- Explain why an index recall of 1.0 in task 4 says nothing about the relevance recall measured in task 1.

**Your answer**